# Figure 3c / 3d -- main-text datasets

Plasschaert, Turtle brain and Mouse hypothalamus: predicted-cluster UMAPs
(3c) and annotated-label UMAPs (3d), one row of six methods per dataset.

Split from `Figure3B+S1A+S2A.ipynb`; the supplementary datasets are in
`figS4_S5.ipynb`. The `## simulation` section of the original
notebook is not part of Figure 3 and stayed only in the original file.


In [2]:
import pandas as pd
import scanpy as sc
import numpy as np
import h5py
import umap
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from matplotlib.pyplot import plot,savefig
from sklearn import metrics

import warnings
warnings.filterwarnings("ignore")


import seaborn as sns

/Volumes/SSD/MCW/Research/Aim 1/DMVAE/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean')

In [5]:
def plot_cluster(df, method_name, y_true, y_true_int, by, ax, y_true2=None, y_true2_int=None):
    """
    df: result object for the method
    method_name: string key in embeddings dict
    y_true: original labels (e.g. strings)
    y_true_int: integer-encoded labels
    by: "pred" or "true"
    ax: matplotlib axis
    """
    if method_name in ('scVI', 'ADClust', 'scAce'):
        y_use_int = y_true_int
    else:
        y_use_int = y_true2_int if y_true2_int is not None else y_true_int

    emb_all = np.asarray(embeddings[method_name])

    if method_name == 'scAce':
        y_pred = df['Clusters'][-1][-1]
    elif method_name == 'ADClust':
        y_pred = df['Clusters']
    else:
        y_pred = df['Clusters']

    y_pred = np.asarray(y_pred, dtype='int').squeeze()
    n = min(len(y_pred), len(y_use_int))
    if len(y_pred) != len(y_use_int):
        y_pred = y_pred[:n]
        emb_all = emb_all[:n]
        y_use_int = y_use_int[:n]
    if isinstance(umap_all, dict) and method_name in umap_all:
        u = umap_all[method_name]
        umap_coords = u[:n] if len(u) > n else u
    else:
        umap_coords = reducer.fit_transform(emb_all)

    if method_name in ('scGMAAE', 'scGNN', 'scDAC', 'DMVAE') or method_name.lower() == 'scvi':
        ari = np.round(df['ARI'], 2) if method_name.lower() != 'scvi' else np.round(df['ARI'], 2)
        nmi = np.round(df['NMI'], 2) if method_name.lower() != 'scvi' else np.round(df['NMI'], 2)
    else:
        ari = np.round(metrics.adjusted_rand_score(y_pred, y_use_int), 2)
        nmi = np.round(metrics.normalized_mutual_info_score(y_pred, y_use_int), 2)
    ari = float(np.atleast_1d(ari).flat[-1])
    nmi = float(np.atleast_1d(nmi).flat[-1])
    print('Method: {}, ARI={}, NMI={}'.format(method_name, ari, nmi))

    adata = sc.AnnData(pd.DataFrame(np.random.rand(len(y_pred), 1)))
    adata.obs['pred'] = y_pred
    adata.obs['pred'] = adata.obs['pred'].astype(str).astype('category')

    adata.obs['true'] = y_use_int
    adata.obs['true'] = adata.obs['true'].astype(str).astype('category')

    '''if method_name == 'scvi':
        adata.obs['true'] = y_true_int_scvi
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')
    else:
        adata.obs['true'] = y_true_int
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')'''

    adata.obsm['X_umap'] = umap_coords

    K_pred = len(np.unique(y_pred))
    K_true = len(np.unique(y_use_int))

    if by == "pred":
        sc.pl.umap(adata, color=['pred'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}   ARI = {:.2f}'.format(K_pred, ari), fontsize=30, family='Arial')
    else:
        sc.pl.umap(adata, color=['true'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}'.format(K_true), fontsize=30, family='Arial')

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot([xmin, xmax], [ymin, ymin], color="black", linewidth=1)
    ax.plot([xmin, xmin], [ymin, ymax], color="black", linewidth=1)
    ax.set_facecolor("white")

# Article

## Plasschaert

In [102]:
fig = plt.figure(figsize=(30, 40))
sub_figs = fig.subfigures(10, 1)
axs = []

for i, sub_fig in enumerate(sub_figs):
    axs.append(sub_fig.subplots(1, 6))

axs = np.array(axs)

In [103]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Plasschaert/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

|S28 (6977,)


In [104]:
scvi = np.load('/Users/enid/Downloads/Plass/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Plass/adclust.npz')
scace = np.load('/Users/enid/Downloads/Plass/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Plass/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/Plass/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/Plass/scgnn.npz')

In [105]:
methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])

scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [106]:
#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_plass.npz", UMAP=umap_all)

In [107]:
for j in range(6):
    axs[0][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[0][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[0][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[0][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[0][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[0][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[0][5])

Method: scVI, ARI=0.2942, NMI=0.5545
Method: scGNN, ARI=0.38, NMI=0.56
Method: ADClust, ARI=0.85, NMI=0.78
Method: scAce, ARI=0.59, NMI=0.62
Method: scDAC, ARI=0.34, NMI=0.57
Method: DMVAE, ARI=0.95, NMI=0.9


## turtle_b

In [108]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/turtle_b/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/turtle_b/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/turtle_b/adclust.npz')
scace = np.load('/Users/enid/Downloads/turtle_b/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/turtle_b/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/turtle_b/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/turtle_b/scgnn.npz')

methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])

#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_turtleb.npz", UMAP=umap_all)

plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[1][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[1][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[1][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[1][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[1][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[1][5])

|S5 (18664,)
scVI
scGNN
ADClust
scAce
scDAC
DMVAE
Method: scVI, ARI=0.4472, NMI=0.734
Method: scGNN, ARI=0.49, NMI=0.48
Method: ADClust, ARI=0.62, NMI=0.73
Method: scAce, ARI=0.7, NMI=0.72
Method: scDAC, ARI=0.38, NMI=0.52
Method: DMVAE, ARI=0.81, NMI=0.7


## mouse_h

In [109]:
def plot_cluster(df, method_name, y_true, y_true_int, by, ax):
    """
    df: result object for the method
    method_name: string key in embeddings dict
    y_true: original labels (e.g. strings)
    y_true_int: integer-encoded labels
    by: "pred" or "true"
    ax: matplotlib axis
    """
    emb_all = np.asarray(embeddings[method_name])

    if method_name == 'scAce':
        y_pred = df['Clusters'][-1][-1]
    elif method_name == 'ADClust':
        y_pred = df['Clusters']
    else:
        y_pred = df['Clusters']

    y_pred = np.asarray(y_pred, dtype='int').squeeze()
    n = min(len(y_pred), len(y_true_int))
    if len(y_pred) != len(y_true_int):
        y_pred = y_pred[:n]
        emb_all = emb_all[:n]
        y_true_int = y_true_int[:n]
    # Use precomputed UMAP if available (e.g. loaded from npz), else recompute
    if isinstance(umap_all, dict) and method_name in umap_all:
        u = umap_all[method_name]
        umap_coords = u[:n] if len(u) > n else u
    else:
        umap_coords = reducer.fit_transform(emb_all)

    if method_name in ('scGMAAE', 'scGNN', 'scDAC', 'DMVAE', "scVI"):
        ari = np.round(df['ARI'], 2)
        nmi = np.round(df['NMI'], 2)
    else:
        ari = np.round(metrics.adjusted_rand_score(y_pred, y_true_int), 2)
        nmi = np.round(metrics.normalized_mutual_info_score(y_pred, y_true_int), 2)
    ari = float(np.atleast_1d(ari).flat[-1])
    nmi = float(np.atleast_1d(nmi).flat[-1])
    print('Method: {}, ARI={}, NMI={}'.format(method_name, ari, nmi))

    adata = sc.AnnData(pd.DataFrame(np.random.rand(len(y_pred), 1)))
    adata.obs['pred'] = y_pred
    adata.obs['pred'] = adata.obs['pred'].astype(str).astype('category')

    if method_name.lower() == 'scvi':
        adata.obs['true'] = y_true_int_scvi[:n] if len(y_true_int_scvi) >= n else y_true_int_scvi
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')
    else:
        adata.obs['true'] = y_true_int
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')

    adata.obsm['X_umap'] = umap_coords

    K_pred = len(np.unique(y_pred))
    K_true = len(np.unique(y_true_int))

    if by == "pred":
        sc.pl.umap(adata, color=['pred'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}   ARI = {:.2f}'.format(K_pred, ari), fontsize=30, family='Arial')
    else:
        sc.pl.umap(adata, color=['true'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}'.format(K_true), fontsize=30, family='Arial')

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot([xmin, xmax], [ymin, ymin], color="black", linewidth=1)
    ax.plot([xmin, xmin], [ymin, ymax], color="black", linewidth=1)
    ax.set_facecolor("white")

In [110]:
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/mouse_h/data.h5')
obs = data_mat['obs']
ds = np.loadtxt("/Volumes/SSD/MCW/Research/Aim 1/Data/mouse_h/data_celltype.txt")
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

ds2= obs['cell_type1']
y_true_bytes_scvi = np.array(ds2)
y_true_scvi = y_true_bytes_scvi.astype(str)
classes_scvi, y_true_int_scvi = np.unique(y_true_scvi, return_inverse=True)

data_mat.close()
scvi = np.load('/Users/enid/Downloads/mouse_h/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/mouse_h/adclust.npz')
scace = np.load('/Users/enid/Downloads/mouse_h/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/mouse_h/dmvae.npz')
scdac = np.load('/Users/enid/Downloads/mouse_h/scdac.npz')
scgnn = np.load('/Users/enid/Downloads/mouse_h/scgnn.npz')

methods = {
    "scVI": scvi,
    "scGNN": scgnn,
    "ADClust": adclust,
    "scAce": scace,
    "scDAC": scdac,
    "DMVAE": dmvae,
}

embeddings = {}
for name, method in methods.items():
    emb = method["Embedding"]
    # 2D = single matrix (scVI, DMVAE, scGNN, scDAC); 3D = use last (ADClust, scAce)
    embeddings[name] = emb if emb.ndim == 2 else emb[-1]

umap_all = {}
for name, emb in embeddings.items():
    print(name)
    adata = sc.AnnData(emb)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata, random_state=0)
    umap_all[name] = np.array(adata.obsm["X_umap"])

#umap_all = np.load("umap/umap_f3_human.npz")['UMAP']
np.savez("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_mouseh.npz", UMAP=umap_all)


float64 (12079,)
scVI
scGNN
ADClust
scAce
scDAC
DMVAE


In [111]:
# Clear last row and redo
for j in range(6):
    axs[2][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "pred", axs[2][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "pred", axs[2][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "pred", axs[2][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "pred", axs[2][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "pred", axs[2][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "pred", axs[2][5])

Method: scVI, ARI=0.53, NMI=0.75
Method: scGNN, ARI=0.37, NMI=0.49
Method: ADClust, ARI=0.78, NMI=0.78
Method: scAce, ARI=0.84, NMI=0.79
Method: scDAC, ARI=0.74, NMI=0.78
Method: DMVAE, ARI=0.87, NMI=0.79


In [112]:
plt.savefig('/Volumes/SSD/MCW/Research/Aim 1/Documents/Paper_draft/papers/rw.png', dpi=300, format='png', bbox_inches='tight')

# True labels

In [113]:
fig = plt.figure(figsize=(30, 40))
sub_figs = fig.subfigures(10, 1)
axs = []

for i, sub_fig in enumerate(sub_figs):
    axs.append(sub_fig.subplots(1, 6))

axs = np.array(axs)

In [ ]:
def plot_cluster(df, method_name, y_true, y_true_int, by, ax):
    """
    df: result object for the method
    method_name: string key in embeddings dict
    y_true: original labels (e.g. strings)
    y_true_int: integer-encoded labels
    by: "pred" or "true"
    ax: matplotlib axis
    """
    #emb_all = np.asarray(embeddings[method_name])
    #umap_coords = reducer.fit_transform(emb_all)
    umap = umap_all[method_name]
    if method_name == 'scAce':
        y_pred = df['Clusters'][-1][-1]
    else:
        y_pred = df['Clusters']

    y_pred = np.asarray(y_pred, dtype='int').squeeze()

    if method_name in ('scGMAAE', 'scGNN', 'scDAC', 'DMVAE') or method_name.lower() == 'scvi':
        ari = np.round(df['ARI'], 2) if method_name.lower() != 'scvi' else np.round(df['ARI'], 2)
        nmi = np.round(df['NMI'], 2) if method_name.lower() != 'scvi' else np.round(df['NMI'], 2)
    else:
        ari = np.round(metrics.adjusted_rand_score(y_pred, y_true_int), 2)
        nmi = np.round(metrics.normalized_mutual_info_score(y_pred, y_true_int), 2)
    ari = float(np.atleast_1d(ari).flat[-1])
    nmi = float(np.atleast_1d(nmi).flat[-1])
    print('Method: {}, ARI={}, NMI={}'.format(method_name, ari, nmi))

    adata = sc.AnnData(pd.DataFrame(np.random.rand(len(y_pred), 1)))
    adata.obs['pred'] = y_pred
    adata.obs['pred'] = adata.obs['pred'].astype(str).astype('category')

    '''adata.obs['true'] = y_true_int
    adata.obs['true'] = adata.obs['true'].astype(str).astype('category')'''

    if method_name == 'scvi':
        adata.obs['true'] = y_true_int_scvi
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')
    else:
        adata.obs['true'] = y_true_int
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')

    adata.obsm['X_umap'] = umap

    K_pred = len(np.unique(y_pred))
    K_true = len(np.unique(y_true_int))

    if by == "pred":
        sc.pl.umap(adata, color=['pred'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}   ARI = {:.2f}'.format(K_pred, ari), fontsize=30, family='Arial')
    else:
        sc.pl.umap(adata, color=['true'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}'.format(K_true), fontsize=30, family='Arial')

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot([xmin, xmax], [ymin, ymin], color="black", linewidth=1)
    ax.plot([xmin, xmin], [ymin, ymax], color="black", linewidth=1)
    ax.set_facecolor("white")

In [115]:
# Use saved UMAP so plots match the saved coordinates (do not re-run the 'umap_all = {}' compute cell after this)
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_plass.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()


In [116]:
scvi = np.load('/Users/enid/Downloads/Plass/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/Plass/adclust.npz')
scace = np.load('/Users/enid/Downloads/Plass/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/Plass/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/Plass/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/Plass/scdac.npz')

data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/Plasschaert/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

|S28 (6977,)


In [117]:
plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[0][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[0][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[0][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[0][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[0][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[0][5])

Method: scVI, ARI=0.2942, NMI=0.5545
Method: scGNN, ARI=0.38, NMI=0.56
Method: ADClust, ARI=0.85, NMI=0.78
Method: scAce, ARI=0.59, NMI=0.62
Method: scDAC, ARI=0.34, NMI=0.57
Method: DMVAE, ARI=0.95, NMI=0.9


In [118]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_turtleb.npz", allow_pickle=True)['UMAP']
umap_all = umap_all.item()
data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/turtle_b/data.h5')
obs = data_mat['obs']
ds = obs['cell_type1']
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

data_mat.close()

scvi = np.load('/Users/enid/Downloads/turtle_b/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/turtle_b/adclust.npz')
scace = np.load('/Users/enid/Downloads/turtle_b/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/turtle_b/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/turtle_b/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/turtle_b/scdac.npz')

plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[1][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[1][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[1][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[1][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[1][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[1][5])

|S5 (18664,)
Method: scVI, ARI=0.4472, NMI=0.734
Method: scGNN, ARI=0.49, NMI=0.48
Method: ADClust, ARI=0.62, NMI=0.73
Method: scAce, ARI=0.7, NMI=0.72
Method: scDAC, ARI=0.38, NMI=0.52
Method: DMVAE, ARI=0.81, NMI=0.7


In [120]:
def plot_cluster(df, method_name, y_true, y_true_int, by, ax):
    """
    df: result object for the method
    method_name: string key in embeddings dict
    y_true: original labels (e.g. strings)
    y_true_int: integer-encoded labels
    by: "pred" or "true"
    ax: matplotlib axis
    """
    emb_all = np.asarray(embeddings[method_name])

    if method_name == 'scAce':
        y_pred = df['Clusters'][-1][-1]
    elif method_name == 'ADClust':
        y_pred = df['Clusters']
    else:
        y_pred = df['Clusters']

    y_pred = np.asarray(y_pred, dtype='int').squeeze()
    n = min(len(y_pred), len(y_true_int))
    if len(y_pred) != len(y_true_int):
        y_pred = y_pred[:n]
        emb_all = emb_all[:n]
        y_true_int = y_true_int[:n]
    # Use precomputed UMAP if available (e.g. loaded from npz), else recompute
    if isinstance(umap_all, dict) and method_name in umap_all:
        u = umap_all[method_name]
        umap_coords = u[:n] if len(u) > n else u
    else:
        umap_coords = reducer.fit_transform(emb_all)

    if method_name in ('scGMAAE', 'scGNN', 'scDAC', 'DMVAE', "scVI"):
        ari = np.round(df['ARI'], 2)
        nmi = np.round(df['NMI'], 2)
    else:
        ari = np.round(metrics.adjusted_rand_score(y_pred, y_true_int), 2)
        nmi = np.round(metrics.normalized_mutual_info_score(y_pred, y_true_int), 2)
    ari = float(np.atleast_1d(ari).flat[-1])
    nmi = float(np.atleast_1d(nmi).flat[-1])
    print('Method: {}, ARI={}, NMI={}'.format(method_name, ari, nmi))

    adata = sc.AnnData(pd.DataFrame(np.random.rand(len(y_pred), 1)))
    adata.obs['pred'] = y_pred
    adata.obs['pred'] = adata.obs['pred'].astype(str).astype('category')

    if method_name.lower() == 'scvi':
        adata.obs['true'] = y_true_int_scvi[:n] if len(y_true_int_scvi) >= n else y_true_int_scvi
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')
    else:
        adata.obs['true'] = y_true_int
        adata.obs['true'] = adata.obs['true'].astype(str).astype('category')

    adata.obsm['X_umap'] = umap_coords

    K_pred = len(np.unique(y_pred))
    K_true = len(np.unique(y_true_int))

    if by == "pred":
        sc.pl.umap(adata, color=['pred'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}   ARI = {:.2f}'.format(K_pred, ari), fontsize=30, family='Arial')
    else:
        sc.pl.umap(adata, color=['true'], ax=ax, show=False, legend_loc=None, size=8)
        ax.set_title('K = {}'.format(K_true), fontsize=30, family='Arial')

    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.plot([xmin, xmax], [ymin, ymin], color="black", linewidth=1)
    ax.plot([xmin, xmin], [ymin, ymax], color="black", linewidth=1)
    ax.set_facecolor("white")

In [121]:
umap_all = np.load("/Volumes/SSD/MCW/Research/Aim 1/Results/umap_mouseh.npz",  allow_pickle=True)['UMAP']
umap_all = umap_all.item()

data_mat = h5py.File('/Volumes/SSD/MCW/Research/Aim 1/Data/mouse_h/data.h5')
obs = data_mat['obs']
ds = np.loadtxt("/Volumes/SSD/MCW/Research/Aim 1/Data/mouse_h/data_celltype.txt")
print(ds.dtype, ds.shape)
y_true_bytes = np.array(ds)
y_true = y_true_bytes.astype(str)
classes, y_true_int = np.unique(y_true, return_inverse=True)

ds2= obs['cell_type1']
y_true_bytes_scvi = np.array(ds2)
y_true_scvi = y_true_bytes_scvi.astype(str)
classes_scvi, y_true_int_scvi = np.unique(y_true_scvi, return_inverse=True)

data_mat.close()
scvi = np.load('/Users/enid/Downloads/mouse_h/scvi.npz', allow_pickle=True)
adclust = np.load('/Users/enid/Downloads/mouse_h/adclust.npz')
scace = np.load('/Users/enid/Downloads/mouse_h/scace.npz', allow_pickle=True)
dmvae = np.load('/Users/enid/Downloads/mouse_h/dmvae.npz')
scgnn = np.load('/Users/enid/Downloads/mouse_h/scgnn.npz')
scdac = np.load('/Users/enid/Downloads/mouse_h/scdac.npz')

# Clear last row and redo
for j in range(6):
    axs[2][j].clear()
plot_cluster(scvi, 'scVI', y_true, y_true_int, "true", axs[2][0])
plot_cluster(scgnn, 'scGNN', y_true, y_true_int, "true", axs[2][1])
plot_cluster(adclust, 'ADClust', y_true, y_true_int, "true", axs[2][2])
plot_cluster(scace, 'scAce', y_true, y_true_int, "true", axs[2][3])
plot_cluster(scdac, 'scDAC', y_true, y_true_int, "true", axs[2][4])
plot_cluster(dmvae, 'DMVAE', y_true, y_true_int, "true", axs[2][5])

float64 (12079,)
Method: scVI, ARI=0.53, NMI=0.75
Method: scGNN, ARI=0.37, NMI=0.49
Method: ADClust, ARI=0.78, NMI=0.78
Method: scAce, ARI=0.84, NMI=0.79
Method: scDAC, ARI=0.74, NMI=0.78
Method: DMVAE, ARI=0.87, NMI=0.79


In [122]:
plt.savefig('/Volumes/SSD/MCW/Research/Aim 1/Documents/Paper_draft/papers/umap_truth.png', dpi=300, format='png', bbox_inches='tight')